# 05 PSO Hyperparameter Tuning (All Datasets)

This notebook tunes CNN hyperparameters with PSO on **all three datasets** (Desharnais, COCOMO-81, China).

**Key features:**
- Enhanced CNN with BatchNormalization, Dropout, multi-layer support
- EarlyStopping + ReduceLROnPlateau callbacks
- Log-transform on effort target for skew correction
- 6D PSO: [filters, kernel_size, dense_units, lr, dropout_rate, num_conv_layers]
- Per-dataset models saved to `models/` for web app

In [6]:
import json
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

root_dir = Path.cwd()
if not (root_dir / "src").exists() and (root_dir.parent / "src").exists():
    root_dir = root_dir.parent

if not (root_dir / "src").exists():
    raise FileNotFoundError("Could not find project root containing 'src' directory")

sys.path.insert(0, str(root_dir))

from src.cnn_model import build_cnn_regressor, reshape_for_cnn, train_cnn_model
from src.evaluate import evaluate_predictions, save_metrics
from src.feature_engineering import log_transform_target, inverse_log_transform
from src.pso_optimizer import decode_cnn_hyperparameters, tune_cnn_with_pso

processed_dir = root_dir / "data" / "processed"
metrics_dir = root_dir / "results" / "metrics"
models_dir = root_dir / "models"
models_dir.mkdir(parents=True, exist_ok=True)

# Load all 3 datasets
dataset_files = {
    "china": "china_processed.csv",       # 499 rows — best for CNN
    "desharnais": "desharnais_processed.csv",  # ~77 rows
    "cocomo81": "cocomo81_processed.csv",     # ~63 rows
}

datasets = {}
for name, fname in dataset_files.items():
    path = processed_dir / fname
    if path.exists():
        datasets[name] = pd.read_csv(path)
        print(f"Loaded {name}: {datasets[name].shape}")

print(f"\n{len(datasets)} datasets ready for PSO tuning")

Loaded china: (499, 19)
Loaded desharnais: (81, 13)
Loaded cocomo81: (63, 20)

3 datasets ready for PSO tuning


In [7]:
# --- Helper: prepare dataset splits ---

def prepare_dataset(df):
    """Split and reshape a dataset for CNN training."""
    target_col = df.columns[-1]
    X = df.drop(columns=[target_col]).values.astype(np.float32)
    y = df[target_col].values.astype(np.float32)
    y_log = log_transform_target(y).astype(np.float32)

    X_train, X_test, y_train, y_test, y_train_log, y_test_log = train_test_split(
        X, y, y_log, test_size=0.2, random_state=42
    )
    X_train_cnn = reshape_for_cnn(X_train)
    X_test_cnn = reshape_for_cnn(X_test)

    return {
        "X_train_cnn": X_train_cnn, "X_test_cnn": X_test_cnn,
        "y_train": y_train, "y_test": y_test,
        "y_train_log": y_train_log, "y_test_log": y_test_log,
        "n_features": X.shape[1],
    }


# --- Helper: train baseline CNN with fixed hyperparams ---

def train_baseline_cnn(splits):
    """Train a baseline CNN and return metrics + model."""
    model = build_cnn_regressor(
        input_length=splits["n_features"],
        filters=32, kernel_size=3, dense_units=64,
        learning_rate=1e-3, dropout_rate=0.2, num_conv_layers=1,
    )
    model, history = train_cnn_model(
        model, splits["X_train_cnn"], splits["y_train_log"],
        splits["X_test_cnn"], splits["y_test_log"],
        epochs=100, batch_size=32, verbose=0, use_callbacks=True,
    )
    preds_log = model.predict(splits["X_test_cnn"], verbose=0).ravel()
    preds = inverse_log_transform(preds_log)
    metrics = evaluate_predictions("cnn_baseline", splits["y_test"], preds)
    return model, history, metrics


print("Helper functions defined.")

Helper functions defined.


In [8]:
# --- Run PSO + baseline CNN on ALL datasets ---

all_cnn_results = []
all_histories = {}
all_hyperparams = {}

for ds_name, df in datasets.items():
    print(f"\n{'='*70}")
    print(f"  Dataset: {ds_name} ({len(df)} samples, {df.shape[1]-1} features)")
    print(f"{'='*70}")

    splits = prepare_dataset(df)

    # 1. Baseline CNN
    print("\n[1/2] Training baseline CNN...")
    baseline_model, baseline_hist, baseline_metrics = train_baseline_cnn(splits)
    baseline_metrics["dataset"] = ds_name
    all_cnn_results.append(baseline_metrics)
    print(f"  Baseline CNN — MAE: {baseline_metrics['mae'].values[0]:.2f}, "
          f"RMSE: {baseline_metrics['rmse'].values[0]:.2f}, "
          f"R²: {baseline_metrics['r2'].values[0]:.4f}")

    # 2. PSO optimization
    print(f"\n[2/2] Running PSO (15 particles × 25 iterations)...")

    # PSO objective function (closure over this dataset's splits)
    def make_objective(s):
        def objective_fn(particles):
            scores = []
            for particle in particles:
                hp = decode_cnn_hyperparameters(particle)
                try:
                    model = build_cnn_regressor(
                        input_length=s["n_features"],
                        filters=hp["filters"],
                        kernel_size=hp["kernel_size"],
                        dense_units=hp["dense_units"],
                        learning_rate=hp["learning_rate"],
                        dropout_rate=hp.get("dropout_rate", 0.2),
                        num_conv_layers=hp.get("num_conv_layers", 1),
                    )
                    model, history = train_cnn_model(
                        model,
                        s["X_train_cnn"], s["y_train_log"],
                        s["X_test_cnn"], s["y_test_log"],
                        epochs=30, batch_size=32, verbose=0, use_callbacks=True,
                    )
                    val_mae = min(history["val_mean_absolute_error"])
                    scores.append(val_mae)
                except Exception:
                    scores.append(1e6)
            return np.array(scores)
        return objective_fn

    lower_bounds = np.array([8,   2,  8,  1e-4, 0.1, 1], dtype=float)
    upper_bounds = np.array([128, 7, 256, 1e-2, 0.5, 3], dtype=float)

    best_cost, best_position = tune_cnn_with_pso(
        objective_fn=make_objective(splits),
        dimensions=6,
        lower_bounds=lower_bounds,
        upper_bounds=upper_bounds,
        n_particles=15,
        iters=25,
    )

    best_hp = decode_cnn_hyperparameters(best_position)
    all_hyperparams[ds_name] = best_hp
    print(f"  Best PSO val MAE (log): {best_cost:.4f}")
    print(f"  Hyperparams: {best_hp}")

    # 3. Train final CNN+PSO with best hyperparams
    pso_model = build_cnn_regressor(
        input_length=splits["n_features"],
        filters=best_hp["filters"],
        kernel_size=best_hp["kernel_size"],
        dense_units=best_hp["dense_units"],
        learning_rate=best_hp["learning_rate"],
        dropout_rate=best_hp.get("dropout_rate", 0.2),
        num_conv_layers=best_hp.get("num_conv_layers", 1),
    )
    pso_model, pso_hist = train_cnn_model(
        pso_model,
        splits["X_train_cnn"], splits["y_train_log"],
        splits["X_test_cnn"], splits["y_test_log"],
        epochs=100, batch_size=32, verbose=0, use_callbacks=True,
    )

    pso_preds_log = pso_model.predict(splits["X_test_cnn"], verbose=0).ravel()
    pso_preds = inverse_log_transform(pso_preds_log)
    pso_metrics = evaluate_predictions("cnn_pso", splits["y_test"], pso_preds)
    pso_metrics["dataset"] = ds_name
    all_cnn_results.append(pso_metrics)

    print(f"  CNN+PSO    — MAE: {pso_metrics['mae'].values[0]:.2f}, "
          f"RMSE: {pso_metrics['rmse'].values[0]:.2f}, "
          f"R²: {pso_metrics['r2'].values[0]:.4f}")

    # Save per-dataset models
    pso_model.save(models_dir / f"cnn_pso_{ds_name}.h5")
    baseline_model.save(models_dir / f"cnn_baseline_{ds_name}.h5")

    all_histories[ds_name] = {
        "baseline": {k: [float(v) for v in vals] for k, vals in baseline_hist.items()},
        "pso": {k: [float(v) for v in vals] for k, vals in pso_hist.items()},
    }

print("\n\nAll datasets complete!")


  Dataset: china (499 samples, 18 features)

[1/2] Training baseline CNN...


2026-04-17 20:44:04,401 - pyswarms.single.global_best - INFO - Optimize for 25 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}


  Baseline CNN — MAE: 1103.52, RMSE: 4461.78, R²: 0.3377

[2/2] Running PSO (15 particles × 25 iterations)...


pyswarms.single.global_best: 100%|██████████|25/25, best_cost=0.316
2026-04-17 22:36:48,196 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 0.31575626134872437, best pos: [4.95399581e+01 3.65802519e+00 6.33170264e+01 8.51021474e-03
 2.21165177e-01 1.88874356e+00]


  Best PSO val MAE (log): 0.3158
  Hyperparams: {'filters': 50, 'kernel_size': 4, 'dense_units': 63, 'learning_rate': 0.008510214741517726, 'dropout_rate': 0.2211651774775311, 'num_conv_layers': 2}


...

  Baseline CNN — MAE: 322.19, RMSE: 642.34, R²: -0.3352

[2/2] Running PSO (15 particles × 25 iterations)...


pyswarms.single.global_best: 100%|██████████|25/25, best_cost=1.16
2026-04-18 01:00:24,854 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 1.1621530055999756, best pos: [7.48012653e+01 5.56902016e+00 1.32174230e+02 8.07267283e-03
 1.63265272e-01 1.91533191e+00]


  Best PSO val MAE (log): 1.1622
  Hyperparams: {'filters': 75, 'kernel_size': 6, 'dense_units': 132, 'learning_rate': 0.008072672826324971, 'dropout_rate': 0.16326527180867714, 'num_conv_layers': 2}


2026-04-18 01:00:36,139 - absl - WARNING - You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 
2026-04-18 01:00:36,258 - absl - WARNING - You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 


  CNN+PSO    — MAE: 270.48, RMSE: 575.72, R²: -0.0726


All datasets complete!


In [9]:
# --- Summary: CNN results across all datasets ---

cnn_comparison = pd.concat(all_cnn_results, ignore_index=True)
cols = ["dataset", "model", "mae", "rmse", "r2", "mape", "mdmre", "pred25"]
cnn_comparison = cnn_comparison[cols].sort_values(["dataset", "rmse"])

print("CNN Baseline vs CNN+PSO (all datasets)")
print("="*70)
cnn_comparison

CNN Baseline vs CNN+PSO (all datasets)


,dataset,model,mae,rmse,r2,mape,mdmre,pred25
0,china,cnn_baseline,1103.515147,4461.780604,0.337689,30.033954,0.232595,53.000000
1,china,cnn_pso,2047.838747,5577.414946,-0.034932,60.519170,0.470122,21.000000
5,cocomo81,cnn_pso,270.475983,575.724052,-0.072598,281.551205,0.928195,15.384615
4,cocomo81,cnn_baseline,322.193253,642.337642,-0.335165,98.698648,0.996362,0.000000
3,desharnais,cnn_pso,1592.690331,3476.033943,0.052979,34.708506,0.135443,64.705882
2,desharnais,cnn_baseline,4534.424807,5772.433553,-1.611618,99.929505,0.999554,0.000000


In [10]:
# --- Save all outputs ---

# 1. CNN comparison metrics
save_metrics(cnn_comparison, file_name="cnn_vs_pso_metrics.csv", output_dir=metrics_dir)
print("Saved: cnn_vs_pso_metrics.csv")

# 2. Best hyperparams per dataset
hp_path = models_dir / "best_hyperparams.json"
with open(hp_path, "w") as f:
    json.dump(all_hyperparams, f, indent=2)
print(f"Saved: {hp_path}")

# 3. Training histories
hist_path = models_dir / "training_histories.json"
with open(hist_path, "w") as f:
    json.dump(all_histories, f)
print(f"Saved: {hist_path}")

# 4. Full comparison: baselines + CNN
baseline_file = metrics_dir / "baseline_metrics.csv"
if baseline_file.exists():
    baselines = pd.read_csv(baseline_file)
    full_comparison = pd.concat([baselines, cnn_comparison], ignore_index=True)
    full_comparison = full_comparison.sort_values(["dataset", "rmse"])
    save_metrics(full_comparison, file_name="full_comparison.csv", output_dir=metrics_dir)
    print("Saved: full_comparison.csv")

    # Show per-dataset winner
    print("\n" + "="*70)
    print("FULL COMPARISON — Best model per dataset (by RMSE)")
    print("="*70)
    for ds in full_comparison["dataset"].unique():
        ds_df = full_comparison[full_comparison["dataset"] == ds].sort_values("rmse")
        best = ds_df.iloc[0]
        print(f"\n  {ds}: {best['model']} (RMSE={best['rmse']:.2f}, R²={best['r2']:.4f})")
        display(ds_df)
else:
    print("\nRun 03_baselines.ipynb first for full comparison")

Saved: cnn_vs_pso_metrics.csv
Saved: c:\Users\kalpi\OneDrive\Desktop\Software-cost-Estimation\models\best_hyperparams.json
Saved: c:\Users\kalpi\OneDrive\Desktop\Software-cost-Estimation\models\training_histories.json
Saved: full_comparison.csv

FULL COMPARISON — Best model per dataset (by RMSE)

  china: linear_regression (RMSE=1296.14, R²=0.9441)


,dataset,model,mae,rmse,r2,mape,mdmre,pred25
0,china,linear_regression,446.369674,1296.143319,0.944108,36.618911,0.112511,75.0
1,china,xgboost,388.871586,1503.300169,0.924814,11.363155,0.072925,91.0
2,china,random_forest,385.270333,1605.223998,0.914273,10.520690,0.078400,98.0
9,china,cnn_baseline,1103.515147,4461.780604,0.337689,30.033954,0.232595,53.0
10,china,cnn_pso,2047.838747,5577.414946,-0.034932,60.519170,0.470122,21.0



  cocomo81: random_forest (RMSE=404.22, R²=0.4712)


,dataset,model,mae,rmse,r2,mape,mdmre,pred25
3,cocomo81,random_forest,236.484410,404.224316,0.471247,198.270999,0.968000,15.384615
4,cocomo81,xgboost,278.135591,522.988085,0.114901,169.994311,0.774751,15.384615
11,cocomo81,cnn_pso,270.475983,575.724052,-0.072598,281.551205,0.928195,15.384615
12,cocomo81,cnn_baseline,322.193253,642.337642,-0.335165,98.698648,0.996362,0.000000
5,cocomo81,linear_regression,1490.513248,1921.078032,-10.942580,5166.806104,24.503235,7.692308



  desharnais: linear_regression (RMSE=1943.91, R²=0.7038)


,dataset,model,mae,rmse,r2,mape,mdmre,pred25
6,desharnais,linear_regression,1435.054687,1943.914123,0.703827,54.219786,0.296057,41.176471
7,desharnais,xgboost,1693.691995,2245.187760,0.604909,57.648496,0.404122,23.529412
8,desharnais,random_forest,1755.944118,2283.387200,0.591351,77.443114,0.343604,23.529412
13,desharnais,cnn_pso,1592.690331,3476.033943,0.052979,34.708506,0.135443,64.705882
14,desharnais,cnn_baseline,4534.424807,5772.433553,-1.611618,99.929505,0.999554,0.000000
